    In this we introduce the modules used for the spin-triplet pairing calculation.

In [1]:
using LinearAlgebra
using PyCall
using Statistics
using CSV
using DataFrames
using Plots
using DelimitedFiles
@pyimport numpy as np

In [2]:
# Importing personal modules
include("src/bdg_utilities.jl")
include("src/general_utils.jl")

Main.GeneralUtils

In [3]:
# Loading personal modules
using .BdGUtilities: generate_t_map, extract_geometry, build_H_twisted, generate_t_matrix
using .GeneralUtils: fermi_fn, show_all, show_matrix, check_rel_tol

using .BdGUtilities: create_bdg_ham_up, create_bdg_ham_dn, create_bdg_ham_full, create_bdg_ham_full_twisted
using .BdGUtilities: sort_evecs, sort_evecs_spinfull 

using .BdGUtilities: calc_delta_spinfull, calc_avg_n_spinfull, run_self_consistency_numpy_spinfull

f = fermi_fn

fermi_fn (generic function with 2 methods)

In [4]:
"""
compute_pairing_correlators(un_up, vn_up, un_dn, vn_dn, E, T, nSites; tol=1e-12)

Compute the real-space anomalous correlators F_{ij}^{sigma sigma'} from a
spinful BdG solution using only the positive-energy states, and report
Pauli-symmetry diagnostics.

Inputs:
- `un_up`, `vn_up`: BdG amplitudes for spin up  (nStates x nSites)
- `un_dn`, `vn_dn`: BdG amplitudes for spin down  (nStates x nSites)
- `E`: positive BdG eigenvalues (length `nStates`)
- `T`: temperature
- `nSites`: number of lattice sites

Keyword arguments:
- `tol`: small number used to avoid division by zero in relative errors

Returns:
- Dictionary with keys: `"upup"`, `"dndn"`, `"updn"`, `"dnup"`, and `"errors"`
"""
function compute_pairing_correlators(un_up, vn_up,
    un_dn, vn_dn,
    E, T, nSites; tol=1e-12)

    nStates = length(E)
    @assert size(un_up, 1) == nStates == size(vn_up, 1) == size(un_dn, 1) == size(vn_dn, 1)
    @assert size(un_up, 2) == nSites == size(vn_up, 2) == size(un_dn, 2) == size(vn_dn, 2)

    Fuu = zeros(ComplexF64, nSites, nSites)
    Fdd = zeros(ComplexF64, nSites, nSites)
    Fud = zeros(ComplexF64, nSites, nSites)
    Fdu = zeros(ComplexF64, nSites, nSites)

    for n in 1:nStates
        w = 1 - 2 * f(E[n]; T=T)

        Fuu .+= w .* (un_up[n, :] * (vn_up[n, :])')
        Fdd .+= w .* (un_dn[n, :] * (vn_dn[n, :])')
        Fud .+= w .* (un_up[n, :] * (vn_dn[n, :])')
        Fdu .+= w .* (un_dn[n, :] * (vn_up[n, :])')
    end

    err_uu = maximum(abs.(Fuu .+ transpose(Fuu))) / max(maximum(abs.(Fuu)), tol)
    err_dd = maximum(abs.(Fdd .+ transpose(Fdd))) / max(maximum(abs.(Fdd)), tol)

    # singlet symmetric; triplet m0 antisymmetric
    Fs = 0.5 .* (Fud .- Fdu)
    Ft0 = 0.5 .* (Fud .+ Fdu)

    err_s = maximum(abs.(Fs .- transpose(Fs))) / max(maximum(abs.(Fs)), tol)   # should be ~0
    err_t0 = maximum(abs.(Ft0 .+ transpose(Ft0))) / max(maximum(abs.(Ft0)), tol)   # should be ~0

    return Dict(
        "upup" => Fuu,
        "dndn" => Fdd,
        "updn" => Fud,
        "dnup" => Fdu,
        "errors" => Dict("err_uu" => err_uu, "err_dd" => err_dd, "err_s" => err_s, "err_t0" => err_t0)
    )
end

compute_pairing_correlators

In [5]:
N             = 5
nSites        = 4 * N ^ 2

nUp          = ones(Float64, nSites)
nDn          = ones(Float64, nSites)

df = CSV.read("data/N$(N)/raw_df_square-octagon$N.csv", DataFrame)

x, y, neighbors = extract_geometry(df)
lambda = 0.2
lambdaIso = 0.0
t2 = 1.0

tMap = generate_t_map(neighbors, t2=t2)

isoMap = nothing

if lambda == 0.0
    deltaOld      = ones(Float64, nSites)
    isComplexCalc = false
else
    deltaOld      = ones(ComplexF64, nSites)
    isComplexCalc = true
end

impuritySite  = 3
mu             = -1.0
U             = 3

J             = 0.0
K             = 0.0
T             = 0.0

tMatFileName = "data/N$(N)/ham_$(N)_t2_$(t2)"
tMat = generate_t_matrix(tMatFileName); ## One doesn't need to use prebuilt hamiltonian. dataframe is enough to construct this matrix

In [6]:
deltaFinal, _, _, nAvg, evecs, evals, isConverged, endTime, count = 
    run_self_consistency_numpy_spinfull(
        deltaOld,
        mu, nSites,
        nUp, nDn,
        tMat, U, J,
        x, y, neighbors,
        lambda, lambdaIso,
        tMap, isoMap,
        impuritySite, T,
    );

In [7]:
un_up, vn_dn, un_dn, vn_up = sort_evecs_spinfull(evecs, evals, nSites)
evalsPositive = evals[2*nSites + 1:end];

In [8]:
F = compute_pairing_correlators(un_up, vn_dn, un_dn, vn_up, evalsPositive, T, nSites)

Dict{String, Any} with 5 entries:
  "errors" => Dict("err_t0"=>1.64888e-14, "err_dd"=>2.0, "err_s"=>2.0, "err_uu"…
  "updn"   => ComplexF64[-9.36161e-18-1.2141e-17im 0.00294753-0.00278663im … 0.…
  "dnup"   => ComplexF64[2.70537e-18+5.15843e-18im 0.00294753+0.00278663im … 0.…
  "dndn"   => ComplexF64[-0.297575-9.39592e-18im 0.0500335-0.000375578im … -0.0…
  "upup"   => ComplexF64[0.297575+3.33106e-18im -0.0500335-0.000375578im … 0.00…

In [9]:
"""
    triplet_diagnostics(F, neighbors)

Bond-resolved odd-parity triplet diagnostics from correlators.

Returns:
- rms_uu, rms_dd, rms_eqspin (uu+dd), rms_mz0, rms_singlet
- bond_map: (i,j,w_eqspin,w_mz0) for j>i bonds
"""
function triplet_diagnostics(F, neighbors)
    Fuu = F["upup"]
    Fdd = F["dndn"]
    Fud = F["updn"]
    Fdu = F["dnup"]

    nSites = length(neighbors)

    acc_uu = 0.0
    acc_dd = 0.0
    acc_eq = 0.0
    acc_0 = 0.0
    acc_s = 0.0
    nbonds = 0

    bond_map = Vector{Tuple{Int,Int,Float64,Float64}}()

    for i in 1:nSites
        for j in neighbors[i]
            if j > i
                # odd-parity equal-spin components
                fuu = (Fuu[i, j] - Fuu[j, i]) / sqrt(2)
                fdd = (Fdd[i, j] - Fdd[j, i]) / sqrt(2)

                w_uu = abs2(fuu)
                w_dd = abs2(fdd)
                w_eq = w_uu + w_dd

                # odd-parity m_z=0 triplet: antisym part of (Fud+Fdu)/2
                f0 = ((Fud[i, j] + Fdu[i, j]) - (Fud[j, i] + Fdu[j, i])) / (2 * sqrt(2))
                w_0 = abs2(f0)

                # singlet on the same bond (symmetric part of (Fud-Fdu)/2)
                fs = ((Fud[i, j] - Fdu[i, j]) + (Fud[j, i] - Fdu[j, i])) / (2 * sqrt(2))
                w_s = abs2(fs)

                acc_uu += w_uu
                acc_dd += w_dd
                acc_eq += w_eq
                acc_0 += w_0
                acc_s += w_s
                nbonds += 1

                push!(bond_map, (i, j, w_eq, w_0))
            end
        end
    end

    denom = nbonds > 0 ? nbonds : 1
    return (
        rms_uu=nbonds > 0 ? sqrt(acc_uu / denom) : 0.0,
        rms_dd=nbonds > 0 ? sqrt(acc_dd / denom) : 0.0,
        rms_eqspin=nbonds > 0 ? sqrt(acc_eq / denom) : 0.0,
        rms_mz0=nbonds > 0 ? sqrt(acc_0 / denom) : 0.0,
        rms_singlet=nbonds > 0 ? sqrt(acc_s / denom) : 0.0,
        bond_map=bond_map
    )
end

triplet_diagnostics

In [10]:
res = triplet_diagnostics(F, neighbors)

(rms_uu = 0.00028399595281296405, rms_dd = 0.0002839959528129718, rms_eqspin = 0.00040163092812716875, rms_mz0 = 0.006608286213935059, rms_singlet = 6.470620399196281e-17, bond_map = [(1, 4, 1.3572739105954824e-7, 3.125721248989405e-5), (1, 2, 5.642358485328914e-7, 1.7375845845669814e-5), (1, 15, 1.3369840507180947e-7, 1.015519731532461e-11), (1, 3, 7.690552474814366e-9, 6.2111210069585095e-9), (2, 3, 1.6433905803794085e-7, 1.1201489239139057e-5), (2, 12, 1.2874870393393217e-6, 0.0008608127283827635), (2, 4, 7.690552474816025e-9, 6.710598997851526e-5), (3, 4, 5.642358485330553e-7, 1.5530579674239862e-5), (3, 17, 1.2874870393393579e-6, 1.561063572902209e-33), (4, 6, 1.3369840507175886e-7, 1.4338503933706295e-5)  …  (93, 95, 2.8759228883259266e-10, 7.207970096111014e-33), (94, 95, 1.7138658621650353e-7, 2.5254983725763008e-5), (94, 96, 2.9050646920026975e-32, 0.00010009094472925975), (95, 96, 1.6816634332258024e-7, 2.5158302363011397e-5), (97, 100, 1.8552970613425593e-7, 2.50658565506129

In [11]:
@show res.rms_uu res.rms_dd

res.rms_uu = 0.00028399595281296405
res.rms_dd = 0.0002839959528129718


0.0002839959528129718

In [12]:
@show res.rms_eqspin

res.rms_eqspin = 0.00040163092812716875


0.00040163092812716875